# RecoMart Data Preparation and EDA

This notebook demonstrates the production preparation pipeline over the latest validated batch. Explicit ratings (1–5) remain separate from implicit weighted feedback. A zero implicit matrix cell or `NaN` explicit rating means no observed interaction—not bad data and not a fabricated rating.

## Environment and Configuration

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.preparation.config import load_preparation_config
from src.preparation.loader import resolve_validated_batch, load_validated_datasets
from src.preparation.runner import PreparationRunner

BATCH_ID = None  # Set an immutable batch ID here when required.
config = load_preparation_config(project_root=ROOT)
batch = resolve_validated_batch(config.validation_report_path, BATCH_ID)
batch.batch_id

## Load Validated Data
Only validation-manifest-declared paths are loaded; quarantine and raw storage are never read.

In [ ]:
validated = load_validated_datasets(batch)
pd.DataFrame([{'dataset': name, 'rows': len(frame), 'columns': len(frame.columns), 'dtypes': frame.dtypes.astype(str).to_dict()} for name, frame in validated.items()])

## Execute Reusable Preparation Logic
The notebook delegates cleaning, encoding, normalization, matrix construction, splitting, EDA, persistence, and lineage to `src/preparation`.

In [ ]:
manifest = PreparationRunner(config).run(batch_id=batch.batch_id, run_eda=True)
manifest

## Cleaning Summary
Required values are never arbitrarily imputed. Optional release dates remain null; unavailable optional popularity enrichment is recorded as a warning. Technical duplicates are removed only if detected and the counts are preserved.

In [ ]:
summary = json.loads(Path(manifest['preparation_summary_path']).read_text())
pd.DataFrame(manifest['cleaning_actions'])

## Interaction, User Activity, Item Popularity, Ratings, Categories, Price, and Timeline

In [ ]:
eda = json.loads(Path(manifest['eda_summary_path']).read_text())
display(pd.Series(eda['interaction_type_counts'], name='events'))
display(pd.Series(eda['rating_distribution'], name='ratings'))
display(pd.DataFrame(eda['top_products']).head(20))
display(pd.DataFrame(eda['top_users']).head(20))

## User–Item Sparsity
`density = observed unique user-product pairs / (users × products)` and `sparsity = 1 - density`. The heatmap is explicitly a top-50 sampled visualization; the complete matrix is stored long-form and represented sparsely in memory.

In [ ]:
pd.Series({key: eda[key] for key in ['users', 'products', 'unique_user_product_pairs', 'possible_user_product_pairs', 'density', 'sparsity']})

## Categorical Encoding and Numerical Normalization
Nominal fields use one-hot encoding with an explicit Unknown category. Brand uses frequency encoding. Continuous variables use StandardScaler; skewed counts use log1p followed by scaling. Metadata makes both transformations reproducible.

In [ ]:
display(pd.DataFrame(manifest['encoding_methods']['one_hot']).T)
display(pd.DataFrame(manifest['normalization_methods']).T)

## Prepared Datasets and Chronological Splits

In [ ]:
outputs = {name: Path(path) for name, path in manifest['output_dataset_paths'].items()}
schemas = []
for name, path in outputs.items():
    if path.suffix == '.parquet':
        frame = pd.read_parquet(path)
        schemas.append({'dataset': name, 'rows': len(frame), 'columns': len(frame.columns), 'sample': frame.head(2).to_dict('records')})
pd.DataFrame(schemas)[['dataset', 'rows', 'columns']]

## Findings and Conclusions

Use the computed EDA summary and saved figures to assess event imbalance, rating concentration, user activity, popularity long tail, sparsity, and chronological cold-start counts. Sparse implicit-feedback methods are appropriate when density is low; explicit ratings remain suitable for rating-aware evaluation. New users/products in later splits must be handled explicitly by downstream models.

In [ ]:
findings = {
    'most_common_interaction': max(eda['interaction_type_counts'], key=eda['interaction_type_counts'].get),
    'density_percent': eda['density'] * 100,
    'sparsity_percent': eda['sparsity'] * 100,
    'cold_start': eda['cold_start'],
    'plots': manifest['plot_paths'],
}
findings